In [ ]:
# Wide & Deep DNN for Reorder Prediction
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers, models, regularizers
from tensorflow.keras.utils import to_categorical

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report, 
    confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Load engineered data
if os.path.exists('processed/full_data_engineered.parquet'):
    df = pd.read_parquet('processed/full_data_engineered.parquet', engine='fastparquet')
else:
    df = pd.read_parquet('processed/full_data.parquet', engine='fastparquet')

print(f"Full data shape: {df.shape}")

# Sample data for faster training (remove this line for full training)
df = df.sample(n=500000, random_state=42)
print(f"Sampled data shape: {df.shape}")

# Clean missing values
df_clean = df.dropna(subset=['reordered'])
print(f"Rows after cleaning: {len(df_clean)}")

In [ ]:
# Define features
sparse_features = ['user_id', 'product_id', 'aisle_id', 'department_id']
numeric_features = [
    'order_frequency', 'avg_basket_size', 'days_since_prior_order',
    'order_dow', 'order_hour_of_day', 'product_reorder_rate', 
    'product_buys_count'
]

# Target
target = 'reordered'

# Prepare data - handle NaNs in sparse features first
X_sparse = df_clean[sparse_features].fillna(-1).astype(int)
X_numeric = df_clean[numeric_features]
y = df_clean[target].astype(int)

# Encode sparse features
encoders = {}
X_sparse_encoded = X_sparse.copy()

for col in sparse_features:
    le = LabelEncoder()
    # Fit on unique values, excluding -1 (NaN placeholder)
    unique_vals = X_sparse[col].unique()
    unique_vals = [v for v in unique_vals if v != -1]
    le.fit(unique_vals)
    # Transform: -1 becomes 0, actual values are shifted by 1
    X_sparse_encoded[col] = X_sparse[col].apply(lambda x: le.transform([x])[0] if x != -1 else 0)
    encoders[col] = le
    print(f"{col}: {len(le.classes_)} unique values")

# Scale numeric features
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)

# Combine
X_all = np.hstack([X_sparse_encoded.values, X_numeric_scaled])

print(f"\nInput shape: {X_all.shape}")
print(f"Target distribution: {y.value_counts(normalize=True).to_dict()}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1765, random_state=42, stratify=y_train
)  # 0.1765 * 0.85 ≈ 0.15 for validation

print(f"\nSplit sizes:")
print(f"Train: {len(X_train):,}")
print(f"Validation: {len(X_val):,}")
print(f"Test: {len(X_test):,}")


In [ ]:
# Define embedding dimensions
embedding_dims = {
    'user_id': 32,
    'product_id': 64,
    'aisle_id': 16,
    'department_id': 8
}

def create_wide_deep_model(numeric_input_dim, embedding_dims, num_sparse_features=4):
    """
    Wide & Deep Neural Network for Reorder Prediction
    
    Wide Path: Linear model for memorization
    Deep Path: Dense network for generalization
    """
    # Inputs
    numeric_input = layers.Input(shape=(numeric_input_dim,), name='numeric_input')
    
    # Sparse inputs (embeddings)
    sparse_inputs = []
    embedding_layers = []
    
    for i, (feat, dim) in enumerate(embedding_dims.items()):
        # Determine vocab size from encoder
        vocab_size = len(encoders[feat].classes_)
        input_layer = layers.Input(shape=(1,), name=f'sparse_{feat}')
        sparse_inputs.append(input_layer)
        
        # Embedding layer
        embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=dim,
            name=f'embedding_{feat}'
        )(input_layer)
        embedding = layers.Flatten()(embedding)
        embedding_layers.append(embedding)

    # Deep Path
    deep_input = layers.Concatenate()(embedding_layers + [numeric_input])
    
    deep_path = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001))(deep_input)
    deep_path = layers.BatchNormalization()(deep_path)
    deep_path = layers.Dropout(0.3)(deep_path)
    
    deep_path = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001))(deep_path)
    deep_path = layers.BatchNormalization()(deep_path)
    deep_path = layers.Dropout(0.3)(deep_path)
    
    deep_path = layers.Dense(64, activation='relu')(deep_path)
    deep_path = layers.Dropout(0.2)(deep_path)

    # Wide Path (simplified - direct connections from sparse features)
    wide_input = layers.Concatenate()(embedding_layers + [numeric_input])
    wide_path = layers.Dense(32, activation='relu')(wide_input)

    # Concatenate Wide + Deep
    combined = layers.Concatenate()([deep_path, wide_path])

    # Output
    output = layers.Dense(1, activation='sigmoid', name='reordered')(combined)

    # Create model
    all_inputs = sparse_inputs + [numeric_input]
    model = keras.Model(inputs=all_inputs, outputs=output)

    return model

# Create model
model = create_wide_deep_model(
    numeric_input_dim=len(numeric_features),
    embedding_dims=embedding_dims,
    num_sparse_features=len(sparse_features)
)

# Compile
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

model.summary()

In [ ]:
# Prepare data for TensorFlow model (inputs as list)
def prepare_tf_data(X_numeric, X_sparse):
    """
    Convert data to format required by model:
    List of [sparse_user, sparse_product, sparse_aisle, sparse_dept, numeric]
    """
    inputs = []
    
    # Add sparse features (first 4 columns of X_all)
    for i in range(len(sparse_features)):
        inputs.append(X_sparse[:, i])
    
    # Add numeric features (last 7 columns)
    inputs.append(X_numeric)
    
    return inputs

# Prepare datasets
X_train_sparse = X_train[:, :len(sparse_features)]
X_train_numeric = X_train[:, len(sparse_features):]

X_val_sparse = X_val[:, :len(sparse_features)]
X_val_numeric = X_val[:, len(sparse_features):]

X_test_sparse = X_test[:, :len(sparse_features)]
X_test_numeric = X_test[:, len(sparse_features):]

# Create TensorFlow datasets
train_inputs = prepare_tf_data(X_train_numeric, X_train_sparse)
val_inputs = prepare_tf_data(X_val_numeric, X_val_sparse)
test_inputs = prepare_tf_data(X_test_numeric, X_test_sparse)

print(f"Train inputs shape: {[x.shape for x in train_inputs]}")
print(f"Val inputs shape: {[x.shape for x in val_inputs]}")
print(f"Test inputs shape: {[x.shape for x in test_inputs]}")

In [ ]:
# Callbacks
callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_auc',
        patience=5,
        restore_best_weights=True,
        mode='max'
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_auc',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        mode='max'
    )
]

# Train
history = model.fit(
    train_inputs,
    y_train.values,
    validation_data=(val_inputs, y_val.values),
    epochs=20,
    batch_size=512,
    callbacks=callbacks_list,
    verbose=1
)

In [ ]:
# Predict on test set
y_pred_proba = model.predict(test_inputs).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

# Metrics
print("="*50)
print("DNN MODEL PERFORMANCE")
print("="*50)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['First Time', 'Reordered']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['First Time', 'Reordered'],
            yticklabels=['First Time', 'Reordered'])
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('dnn_confusion_matrix.png', dpi=100)
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, label=f'DNN (AUC = {roc_auc_score(y_test, y_pred_proba):.4f})', color='blue')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Training History
if 'history' in locals():
    axes[1].plot(history.history['auc'], label='Train AUC', color='blue')
    axes[1].plot(history.history['val_auc'], label='Val AUC', color='orange')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('AUC')
    axes[1].set_title('Training History')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dnn_training_history.png', dpi=100)
plt.show()

# Save model
model.save('dnn_reorder_model.h5')
print("Model saved to dnn_reorder_model.h5")

In [ ]:
# Baseline comparison (Logistic Regression)
from sklearn.linear_model import LogisticRegression

print("Training Logistic Regression baseline...")
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_pred_lr_proba = lr.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print("LOGISTIC REGRESSION BASELINE")
print("="*50)
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_lr_proba):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")

# Comparison
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
print(f"\n{'Metric':<15} {'Logistic Reg':<15} {'DNN':<15} {'Improvement':<15}")
print("-" * 60)
metrics = {
    'Accuracy': (accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred)),
    'AUC-ROC': (roc_auc_score(y_test, y_pred_lr_proba), roc_auc_score(y_test, y_pred_proba)),
    'F1-Score': (f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred)),
    'Precision': (precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred)),
    'Recall': (recall_score(y_test, y_pred_lr), recall_score(y_test, y_pred))
}

for metric, (lr_val, dnn_val) in metrics.items():
    improvement = dnn_val - lr_val
    print(f"{metric:<15} {lr_val:<15.4f} {dnn_val:<15.4f} {improvement:<+15.4f}")